# Cold start — 26 октября 2026

Пользователь согласовал локальный общий корпус за три недели и окно персонализации 21 день; персонализация после регистрации. Архитектура и действующие издатели не меняются.

Получено 352 реальных материала: ЦБ 11, Frank Media 341. Источник подтверждения — публичные RSS, SHA-256 каждого ответа и время окна в `../backend/coldstart/0008_news_coldstart.manifest.json`. Сырые ответы: ignored `../data/coldstart-2026-10-26/`. Frank Media: шесть RSS-страниц, самая старая доходит до 2 октября; RSS ЦБ доходит до 29 сентября. Это покрытие полученных RSS, не доказательство полноты всех публикаций сайтов.

Интерфакс: неоднократный HTTPS connection timeout, включая IPv4; 0 загруженных публикаций. Frank Media выдаёт HTML challenge для robots.txt, публичный RSS доступен; обход защиты и загрузка страниц статей не выполнялись. robots.txt ЦБ получен.

Снимок переносится через встроенный SQL и существующий Store.Migrate. Исходные даты/ID сохранены. Более свежие версии и завершённые задания не перезаписываются. При позднем деплое даты не сдвигаются. Модельные ответы/секреты в снимок не входят.

Локальная PostgreSQL в отдельном контейнере `alpha-coldstart-postgres`, порт 55432, БД `alpha_coldstart`; контейнер и данные оставлены для продолжения. Код ниже выполнен основной моделью на этой БД. Для повтора явно задайте COLDSTART_DATABASE_URL, значение подключения не записано в notebook.


In [1]:
import os, json
import psycopg
with psycopg.connect(os.environ["COLDSTART_DATABASE_URL"]) as db:
    rows = db.execute("SELECT source_id,count(*),min(published_at),max(published_at),count(*) FILTER(WHERE body <> '') FROM news_items GROUP BY source_id ORDER BY source_id").fetchall()
    result = [dict(source_id=s, count=n, first=first.isoformat(), last=last.isoformat(), with_body=b) for s,n,first,last,b in rows]
    result.append(dict(users=db.execute("SELECT count(*) FROM users").fetchone()[0], deliveries=db.execute("SELECT count(*) FROM news_deliveries").fetchone()[0]))
print(json.dumps(result, ensure_ascii=False, indent=2))


[
  {
    "source_id": "S077",
    "count": 11,
    "first": "2026-10-05T07:40:00+00:00",
    "last": "2026-10-23T10:30:00+00:00",
    "with_body": 0
  },
  {
    "source_id": "S142",
    "count": 341,
    "first": "2026-10-04T21:02:00+00:00",
    "last": "2026-10-26T09:15:22+00:00",
    "with_body": 341
  },
  {
    "users": 0,
    "deliveries": 0
  }
]


## Фактические проверки

- RED: `TestColdStartNewUserAndRoleChangeUseThreeWeeks` воспроизвёл отсутствие задания для 14-дневной новости; после замены окна на 504h GREEN. 22-дневная новость исключена, смена роли повторно ставит 14-дневную в очередь.
- `TestColdStartSnapshotReplayPreservesLiveRevision`: повторное применение реального seed не меняет число новостей, не перезаписывает живой текст и не возобновляет завершённое задание.
- RED/GREEN `TestColdStartDefaultsAndOptOut`: 21 день по умолчанию и NEWS_COLDSTART_ENABLED=false для изолированных установок.
- Финальный `go test -race ./... -count=1 -json`: 128 run, 128 pass, 0 fail, 0 skip (с подслучаями). Три отдельные локальные тестовые БД, архивная БД не использовалась. Исходный журнал: `/tmp/alpha-coldstart-tools/go-tests.jsonl`. `go vet ./...`: exit 0.
- `python -m unittest scripts.test_prepare_news_coldstart`: 4 теста, OK. Хеш, состав, идентичность и временное окно снимка проверены; входы с HTML challenge, неверной датой/доменом отвергаются.
- `scripts/test_unified.py`: status passed, 2 компании, 1 строка ленты, 6 **синтетических** модельных вызовов; после перезапуска чтение не вызывает модель; чужая бизнес-ситуация вернула 404. Журнал: `/tmp/alpha-coldstart-tools/unified.log`. Это не реальные LLM-результаты.
- `git diff --check`: exit 0. Коммит, push и deploy не выполнялись.

Платные LLM-вызовы при предзагрузке не выполнялись. Число пользователей/заданий ниже равно нулю, поскольку согласован общий корпус до регистрации. Наличие новостей не подтверждает наличие персональных инсайтов. Архив Интерфакса остаётся незагруженным. Блок пользователем не закрыт.
